# PIMMUR multi-agent simulation tutorial

This Colab notebook runs a small **Social Balance** simulation from the PIMMUR multi-agent system (MAS). It uses three agents, two conversation rounds, and one simulation run so you can verify the workflow before increasing the scale.


## 1. Get the code and install dependencies

Run this once in a fresh Colab runtime.

In [ ]:
!git clone https://github.com/JXZhou0224/PIMMUR.git
%cd /content/PIMMUR
%pip install -q -r requirements.txt
%pip install -q -r requirements.txt 'httpx<0.28'
import nltk
nltk.download('vader_lexicon', quiet=True)

## 2. Provide an API key

The bundled Social Balance configuration uses the `openai` provider. Please set your API key in `Secrets` on the left of the Colab panel as Name: `OPENAI_API_KEY` and Value: `<your openai api key>` make sure to allow Notebook access


In [ ]:
from google.colab import userdata
import os
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 3. Inspect an existing configuration

PIMMUR reads a JSON configuration. This tutorial uses the repository's existing Social Balance configuration without changing the simulation code or creating a custom configuration. Review the agent count, number of simulations, and rounds before running it.

In [ ]:
%cd /content/PIMMUR/MAS

from pathlib import Path

config_path = Path('configs/SocialBalance.json')
print(config_path.read_text())

## 4. Run the simulation

`run.py` loads the existing configuration, creates the requested agents, runs the conversation rounds, and writes `result.csv` plus `log.jsonl` under `result/social_balance/`. The bundled configuration requests 64 simulations, so this can use substantial API quota. By running the line below, we can set the simulation count to 5, serving as a minimum demo.

In [ ]:
!sed -i 's/"simulation_n": 64/"simulation_n": 5/' configs/SocialBalance.json

Now we can run the experiment:

In [ ]:
!rm -rf result/social_balance
!mkdir -p result/social_balance # clean the previous results
!python run.py --config configs/SocialBalance.json

## 5. Inspect the output

`result.csv` contains the final relation state for each completed simulation. `log.jsonl` records the relation trajectory by round.
for easy representation the 0 means friend and 2 means enemy

In [ ]:
import pandas as pd

result_dir = Path('result/social_balance')
display(pd.read_csv(result_dir / 'result.csv'))
print((result_dir / 'log.jsonl').read_text().splitlines()[0])

## 6. Visualize the Social Balance result

Run the repository's existing visualization script from its result directory. It reads `result.csv` and displays the Friends, Enemies, and Mixed case counts.

In [ ]:
import os
import runpy

previous_dir = Path.cwd()
os.chdir(result_dir)
try:
    runpy.run_path('vis.py')
finally:
    os.chdir(previous_dir)

## Next steps

To run another bundled experiment, use `configs/HerdEffect.json` or `configs/NetworkGrowth.json`. To customize a configuration, make a copy, retain a supported `name` such as `social_balance`, and be aware that results are written under `result/<name>/`.